In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-13 20:28:32,584]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 50

# Multi-seed setting
SEEDS = [42]
patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# TimeSformer Layer-wise Linear Probe | Multi-seed
# ============================================================
from vit_probe_epoch50_patience10 import TimeSformer, TimeSformerLinearProbe

backbone_checkpoint_path = './save_timesformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_timesformer_folder/timesformer_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
criterion = torch.nn.L1Loss().to(device)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mae = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(0):
    for seed in SEEDS:
        print('\n' + '=' * 80)
        print(f'Start TimeSformer probe | seed = {seed}')
        print('=' * 80)
        set_seed(seed)
        train_generator = torch.Generator(); train_generator.manual_seed(seed)
        val_generator = torch.Generator(); val_generator.manual_seed(seed + 10000)
        train_dataloader = DataLoader(dataset=train_set_data,batch_size=batchsz,shuffle=True,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=train_generator)
        val_dataloader = DataLoader(dataset=val_set_data,batch_size=batchsz,shuffle=False,num_workers=num_workerssz,worker_init_fn=seed_worker,generator=val_generator)

        backbone = load_backbone_checkpoint(TimeSformer(), backbone_checkpoint_path)
        model = TimeSformerLinearProbe(backbone=backbone, num_outputs=5).to(device)
        optimizer = torch.optim.AdamW(model.probe_heads.parameters(), lr=lr)
        probe_names = list(model.probe_heads.keys())
        seed_best = {name:{'mae':float('inf'),'epoch':0} for name in probe_names}
        seed_best_mean_mae = float('inf'); no_improve_count = 0
        print('Backbone trainable params:', sum(p.numel() for p in model.backbone.parameters() if p.requires_grad))
        print('Probe dimensions:', model.probe_dims)

        for epoch in range(epochs):
            model.train()
            train_abs_sum = {name:0.0 for name in probe_names}; train_count = {name:0 for name in probe_names}
            for audio, fullshot, big_five_data in tqdm(train_dataloader, desc=f'Probe Seed {seed} Train {epoch+1}/{epochs}'):
                fullshot = fullshot.permute(0,4,1,2,3)
                big_five_data = big_five_data.permute(0,2,1).squeeze()
                audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                optimizer.zero_grad(); outputs = model(fullshot,audio)
                losses = {name:criterion(outputs[name],big_five_data) for name in probe_names}
                torch.stack([losses[name] for name in probe_names]).mean().backward(); optimizer.step()
                with torch.no_grad():
                    for name in probe_names:
                        train_abs_sum[name] += (outputs[name]-big_five_data).abs().sum().item(); train_count[name] += big_five_data.numel()
            train_mae = {name:train_abs_sum[name]/train_count[name] for name in probe_names}

            model.eval(); val_abs_sum = {name:0.0 for name in probe_names}; val_count = {name:0 for name in probe_names}
            with torch.no_grad():
                for audio, fullshot, big_five_data in tqdm(val_dataloader, desc=f'Probe Seed {seed} Val {epoch+1}/{epochs}'):
                    fullshot = fullshot.permute(0,4,1,2,3)
                    big_five_data = big_five_data.permute(0,2,1).squeeze()
                    audio, fullshot, big_five_data = audio.to(device), fullshot.to(device), big_five_data.to(device)
                    outputs = model(fullshot,audio)
                    for name in probe_names:
                        val_abs_sum[name] += (outputs[name]-big_five_data).abs().sum().item(); val_count[name] += big_five_data.numel()
            val_mae = {name:val_abs_sum[name]/val_count[name] for name in probe_names}
            mean_val_mae = float(np.mean([val_mae[name] for name in probe_names]))
            for name in probe_names:
                if val_mae[name] < seed_best[name]['mae']:
                    seed_best[name]['mae'] = float(val_mae[name]); seed_best[name]['epoch'] = epoch + 1
            if mean_val_mae < seed_best_mean_mae:
                seed_best_mean_mae = mean_val_mae; no_improve_count = 0
                if mean_val_mae < global_best_mean_mae:
                    global_best_mean_mae = mean_val_mae; global_best_seed = seed; global_best_epoch = epoch + 1
                    torch.save({'seed':seed,'epoch':epoch+1,'probe_model':model.probe_heads.state_dict(),'mean_val_mae':mean_val_mae,'per_probe_val_mae':val_mae,'probe_dims':model.probe_dims},probe_best_model_path)
            else: no_improve_count += 1

            print(f'\nSeed {seed} | Epoch {epoch+1:03d} | Mean Val MAE={mean_val_mae:.6f} | Patience={no_improve_count}/{patience}')
            print('{:<18} {:>12} {:>12} {:>12}'.format('Representation','Train MAE','Val MAE','1-MAE'))
            print('-'*58)
            for name in probe_names: print('{:<18} {:>12.6f} {:>12.6f} {:>12.6f}'.format(name,train_mae[name],val_mae[name],1.0-val_mae[name]))
            if no_improve_count >= patience:
                logger.info('Probe early stopping | seed={} | epoch={} | best mean MAE={:.6f}'.format(seed,epoch+1,seed_best_mean_mae)); break
            torch.cuda.empty_cache()

        seed_results.append({'seed':seed,'representations':{name:{'best_epoch':seed_best[name]['epoch'],'best_mae':seed_best[name]['mae'],'best_1_mae':1.0-seed_best[name]['mae']} for name in probe_names}})
        del model, backbone, optimizer, train_dataloader, val_dataloader
        gc.collect(); torch.cuda.empty_cache()

print('\n' + '='*90)
print('FINAL TIMESFORMER LAYERWISE LINEAR PROBE RESULTS')
print('='*90)
print('{:<18} {:>14} {:>14} {:>14} {:>14}'.format('Representation','MAE Mean','MAE Std','1-MAE Mean','1-MAE Std'))
print('-'*90)
for name in probe_names:
    maes = np.array([r['representations'][name]['best_mae'] for r in seed_results],dtype=np.float64); one_maes = 1.0-maes
    print('{:<18} {:>14.6f} {:>14.6f} {:>14.6f} {:>14.6f}'.format(name,maes.mean(),maes.std(ddof=1),one_maes.mean(),one_maes.std(ddof=1)))
print('-'*90)
print('Global best mean-probe checkpoint: seed={}, epoch={}, mean MAE={:.6f}'.format(global_best_seed,global_best_epoch,global_best_mean_mae))
print('Saved probe checkpoint:',probe_best_model_path)
print('='*90)



Start TimeSformer probe | seed = 42
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
Backbone trainable params: 0
Probe dimensions: {'block1': 768, 'block2': 768, 'block3': 768, 'block4': 768, 'block5': 768, 'block6': 768, 'block7': 768, 'block8': 768, 'block9': 768, 'block10': 768, 'block11': 768, 'block12': 768, 'final_video': 768, 'final_cross': 768, 'final_fused': 768}


Probe Seed 42 Train 1/50: 100%|█████████▉| 1497/1498 [02:05<00:00, 12.88it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
Probe Seed 42 Val 1/50: 100%|██████████| 498/498 [00:47<00:00, 10.51it/s]



Seed 42 | Epoch 001 | Mean Val MAE=0.088075 | Patience=0/10
Representation        Train MAE      Val MAE        1-MAE
----------------------------------------------------------
block1                 0.098233     0.099908     0.900092
block2                 0.080398     0.089257     0.910743
block3                 0.072334     0.089049     0.910951
block4                 0.116360     0.087711     0.912289
block5                 0.088184     0.087598     0.912402
block6                 0.068758     0.088335     0.911665
block7                 0.087452     0.088047     0.911953
block8                 0.064439     0.087915     0.912085
block9                 0.079066     0.087832     0.912168
block10                0.111468     0.087986     0.912014
block11                0.086995     0.088042     0.911958
block12                0.090936     0.087100     0.912900
final_video            0.100240     0.085318     0.914682
final_cross            0.099969     0.082083     0.917917
final_fuse

Probe Seed 42 Val 2/50: 100%|██████████| 498/498 [00:47<00:00, 10.38it/s]



Seed 42 | Epoch 002 | Mean Val MAE=0.087339 | Patience=0/10
Representation        Train MAE      Val MAE        1-MAE
----------------------------------------------------------
block1                 0.049270     0.094435     0.905565
block2                 0.032004     0.088489     0.911511
block3                 0.029195     0.088444     0.911556
block4                 0.028144     0.087889     0.912111
block5                 0.027579     0.087305     0.912695
block6                 0.027830     0.088331     0.911669
block7                 0.027577     0.087304     0.912696
block8                 0.027011     0.087109     0.912891
block9                 0.027226     0.087186     0.912814
block10                0.027451     0.087051     0.912949
block11                0.027038     0.087008     0.912992
block12                0.026383     0.087060     0.912940
final_video            0.023563     0.085269     0.914731
final_cross            0.017511     0.082229     0.917771
final_fuse

...

Probe Seed 42 Val 29/50: 100%|██████████| 498/498 [00:48<00:00, 10.25it/s]



Seed 42 | Epoch 029 | Mean Val MAE=0.085934 | Patience=9/10
Representation        Train MAE      Val MAE        1-MAE
----------------------------------------------------------
block1                 0.032366     0.087872     0.912128
block2                 0.026442     0.087118     0.912882
block3                 0.025167     0.086517     0.913483
block4                 0.024776     0.086547     0.913453
block5                 0.024540     0.086249     0.913751
block6                 0.024381     0.086331     0.913669
block7                 0.023997     0.086116     0.913884
block8                 0.023963     0.086134     0.913866
block9                 0.023868     0.086141     0.913859
block10                0.023797     0.086049     0.913951
block11                0.023713     0.085957     0.914043
block12                0.023720     0.086081     0.913919
final_video            0.021821     0.085025     0.914975
final_cross            0.013521     0.082059     0.917941
final_fuse

Probe Seed 42 Val 30/50: 100%|██████████| 498/498 [00:49<00:00, 10.15it/s]
[2026-09-13 21:56:54,355]::2637483742::INFO::Probe early stopping | seed=42 | epoch=30 | best mean MAE=0.085766



Seed 42 | Epoch 030 | Mean Val MAE=0.085843 | Patience=10/10
Representation        Train MAE      Val MAE        1-MAE
----------------------------------------------------------
block1                 0.032371     0.088180     0.911820
block2                 0.026520     0.087169     0.912831
block3                 0.025256     0.086437     0.913563
block4                 0.024863     0.086202     0.913798
block5                 0.024636     0.086087     0.913913
block6                 0.024443     0.086211     0.913789
block7                 0.024108     0.086054     0.913946
block8                 0.024063     0.085918     0.914082
block9                 0.023945     0.085938     0.914062
block10                0.023883     0.086092     0.913908
block11                0.023798     0.085950     0.914050
block12                0.023824     0.085875     0.914125
final_video            0.021933     0.084808     0.915192
final_cross            0.013483     0.082022     0.917978
final_fus

/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
